# JPMorgan Natural Gas Storage Pricing – Task 2

This notebook prices a natural gas storage contract using historical price data.

In [14]:

import pandas as pd
from datetime import datetime


In [15]:

df = pd.read_csv('Nat_Gas (1).csv')
df['Dates'] = pd.to_datetime(df['Dates'])
df.set_index('Dates', inplace=True)
df.head()




/tmp/ipython-input-3077845479.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dates'] = pd.to_datetime(df['Dates'])


,Prices
Dates,
2020-10-31,10.1
2020-11-30,10.3
2020-12-31,11.0
2021-01-31,10.9
2021-02-28,10.9


In [16]:

def price_gas_storage(
    injection_dates,
    withdrawal_dates,
    prices,
    injection_rate,
    withdrawal_rate,
    max_storage,
    storage_cost_per_day,
    inj_withdraw_cost_per_unit
):
    storage_level = 0
    contract_value = 0

    for date in injection_dates:
        volume = min(injection_rate, max_storage - storage_level)
        contract_value -= volume * prices.loc[date, 'Prices']
        contract_value -= volume * inj_withdraw_cost_per_unit
        storage_level += volume

    for date in withdrawal_dates:
        volume = min(withdrawal_rate, storage_level)
        contract_value += volume * prices.loc[date, 'Prices']
        contract_value -= volume * inj_withdraw_cost_per_unit
        storage_level -= volume

    total_days = (max(withdrawal_dates) - min(injection_dates)).days
    contract_value -= total_days * storage_cost_per_day

    return contract_value


In [17]:
# Sample test dates
injection_dates = [
    pd.Timestamp('2023-05-31'),
    pd.Timestamp('2023-06-30')
]

withdrawal_dates = [
    pd.Timestamp('2023-08-31'),
    pd.Timestamp('2023-09-30')
]

value = price_gas_storage(
    injection_dates,
    withdrawal_dates,
    df,
    injection_rate=500_000,
    withdrawal_rate=500_000,
    max_storage=1_000_000,
    storage_cost_per_day=1000,
    inj_withdraw_cost_per_unit=0.01
)

value

np.float64(108000.0)

In [18]:
print(f"'2023-06-01' in df index: {'2023-06-01' in df.index}")
print(f"'2023-12-01' in df index: {'2023-12-01' in df.index}")

'2023-06-01' in df index: False
'2023-12-01' in df index: False
